# Manage rule sets

List and delete SHACL rule sets (and their versions) in the CDF project, plus the data products that reference them, stuck historic queue jobs, monitor schedules, and stale data-quality workflows left after a view or data-product switch.

Rule sets cannot be updated in place, and CDF refuses to delete a rule set that still has versions. When you change a rule set (for example its `name`), `cdf deploy` fails with `Rule set '...' can not be deleted, because it contains versions`. Delete the rule set here, then run `cdf deploy` again.

CDF also refuses to delete a rule set version that a data product version references. `delete_rulesets` therefore deletes those data product versions first.

Credentials come from the Toolkit `.env` next to `cdf.toml` (`CDF_*` / `IDP_*`), same as `deploy_infrastructure.py`.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

SCRIPTS_DIR = next(
    path for path in (Path.cwd(), Path.cwd() / "modules/common/cdf_dq_runtime/scripts") if (path / "_cli.py").is_file()
)
sys.path.insert(0, str(SCRIPTS_DIR))

from _cli import (  # noqa: E402
    configure_deploy_cli,
    create_client_from_env,
    find_toolkit_project_root,
    load_project_env,
    load_variables,
)

configure_deploy_cli()
print(f"Credentials: {load_project_env()}")
client = create_client_from_env()
API_URL = f"/api/v1/projects/{client.config.project}"
RULESETS_URL = f"{API_URL}/rulesets"
DATA_PRODUCTS_URL = f"{API_URL}/dataproducts"
ALPHA_HEADERS = {"cdf-version": "alpha"}

_project_root = find_toolkit_project_root()
_toolkit_config = None
if _project_root is not None:
    for _name in ("config.dev.yaml", "config.local.yaml", "config.prod.yaml"):
        _candidate = _project_root / _name
        if _candidate.is_file():
            _toolkit_config = _candidate
            break
DATA_QUALITY_SPACE = load_variables(_toolkit_config).get("dataQualitySpace") or "dataQuality"
print(f"Project: {client.config.project}")
print(f"dataQualitySpace: {DATA_QUALITY_SPACE}")

In [ ]:
def _list_items(url: str, limit: int = 100) -> list[dict]:
    items: list[dict] = []
    params: dict[str, object] = {"limit": limit}
    while True:
        body = client.get(url, params=params, headers=ALPHA_HEADERS).json()
        items.extend(body.get("items", []))
        if not body.get("nextCursor"):
            return items
        params["cursor"] = body["nextCursor"]


def _post(url: str, items: list[dict]) -> None:
    client.post(url, json={"items": items}, headers=ALPHA_HEADERS)


def _prefix(dry_run: bool) -> str:
    return "[DRY RUN] Would delete" if dry_run else "Deleted"


def list_ruleset_versions(external_id: str) -> list[str]:
    """Return the versions of a rule set."""
    return [str(item["version"]) for item in _list_items(f"{RULESETS_URL}/{external_id}/versions")]


def list_rulesets() -> list[dict]:
    """Print and return all rule sets in the project with their versions."""
    rulesets = _list_items(RULESETS_URL)
    for ruleset in rulesets:
        ruleset["versions"] = list_ruleset_versions(ruleset["externalId"])
        print(f"{ruleset['externalId']} | {ruleset.get('name')} | versions: {ruleset['versions']}")
    print(f"{len(rulesets)} rule set(s)")
    return rulesets


def list_data_product_versions(external_id: str) -> list[dict]:
    """Return the versions of a data product, including their `quality.rules` references."""
    return _list_items(f"{DATA_PRODUCTS_URL}/{external_id}/versions", limit=10)


def list_data_products() -> list[dict]:
    """Print and return all data products with their versions and referenced rule set versions."""
    data_products = _list_items(DATA_PRODUCTS_URL)
    for data_product in data_products:
        data_product["versions"] = list_data_product_versions(data_product["externalId"])
        print(f"{data_product['externalId']} | {data_product.get('name')}")
        for version in data_product["versions"]:
            rules = [f"{rule['externalId']}:{rule['version']}" for rule in version.get("quality", {}).get("rules", [])]
            print(f"    {version['version']} ({version.get('status')}) | rules: {rules}")
    print(f"{len(data_products)} data product(s)")
    return data_products


def data_product_versions_referencing(ruleset_ids: list[str]) -> list[tuple[str, str]]:
    """Return (data product, version) pairs whose `quality.rules` reference any of the rule sets."""
    referencing: list[tuple[str, str]] = []
    for data_product in _list_items(DATA_PRODUCTS_URL):
        for version in list_data_product_versions(data_product["externalId"]):
            rules = version.get("quality", {}).get("rules", [])
            if any(rule["externalId"] in ruleset_ids for rule in rules):
                referencing.append((data_product["externalId"], version["version"]))
    return referencing


def delete_data_product_versions(ids: list[tuple[str, str]], dry_run: bool = True) -> None:
    """Delete data product versions given as (data product, version) pairs."""
    for external_id, version in ids:
        if not dry_run:
            _post(f"{DATA_PRODUCTS_URL}/{external_id}/versions/delete", [{"version": version}])
        print(f"{_prefix(dry_run)} data product version {external_id}:{version}")


def delete_data_products(external_ids: list[str], dry_run: bool = True) -> None:
    """Delete data products, removing all their versions first, and confirm they are gone."""
    if not external_ids:
        print("No data products selected.")
        return
    for external_id in external_ids:
        versions = [(external_id, version["version"]) for version in list_data_product_versions(external_id)]
        delete_data_product_versions(versions, dry_run)
        if not dry_run:
            _post(f"{DATA_PRODUCTS_URL}/delete", [{"externalId": external_id}])
        print(f"{_prefix(dry_run)} data product {external_id}")
    if dry_run:
        return
    remaining = {data_product["externalId"] for data_product in _list_items(DATA_PRODUCTS_URL)}
    for external_id in external_ids:
        status = "still exists in CDF" if external_id in remaining else "confirmed deleted from CDF"
        print(f"Data product {external_id}: {status}")


def delete_rulesets(external_ids: list[str], dry_run: bool = True) -> None:
    """Delete rule sets with all their versions, and the data product versions that reference them.

    Args:
        external_ids: Rule sets to delete.
        dry_run: Only print what would be deleted.
    """
    delete_data_product_versions(data_product_versions_referencing(external_ids), dry_run)
    for external_id in external_ids:
        for version in list_ruleset_versions(external_id):
            if not dry_run:
                _post(f"{RULESETS_URL}/{external_id}/versions/delete", [{"version": version}])
            print(f"{_prefix(dry_run)} rule set version {external_id}:{version}")
        if not dry_run:
            _post(f"{RULESETS_URL}/delete", [{"externalId": external_id}])
        print(f"{_prefix(dry_run)} rule set {external_id}")

## Select and delete rule sets and data products

The cell lists the rule sets and data products in CDF and shows them as selection lists (Ctrl/Shift-click to select several).

- **Rule sets:** deletes all versions and the rule set. Data product versions that reference them are deleted first.
- **Data products:** deletes all versions and the data product, for example one no longer deployed by this module.

Keep **Dry run** checked to review what would be deleted, then uncheck it and click **Delete selected** again. After a real delete the lists refresh from CDF. Run `cdf deploy` afterwards to recreate what this module defines.

In [ ]:
layout = widgets.Layout(width="95%", height="160px")
style = {"description_width": "110px"}
ruleset_select = widgets.SelectMultiple(description="Rule sets", layout=layout, style=style)
data_product_select = widgets.SelectMultiple(description="Data products", layout=layout, style=style)
dry_run_box = widgets.Checkbox(value=True, description="Dry run (only show what would be deleted)", style=style)
delete_button = widgets.Button(description="Delete selected", button_style="danger", icon="trash")
output = widgets.Output()


def refresh() -> None:
    """List rule sets and data products in CDF and offer them for selection."""
    rulesets = list_rulesets()
    print()
    data_products = list_data_products()
    ruleset_select.options = [
        (f"{ruleset['externalId']} | {ruleset.get('name')} | versions: {ruleset['versions']}", ruleset["externalId"])
        for ruleset in rulesets
    ]
    data_product_select.options = [
        (
            f"{data_product['externalId']} | {data_product.get('name')} | "
            f"versions: {[version['version'] for version in data_product['versions']]}",
            data_product["externalId"],
        )
        for data_product in data_products
    ]


def on_delete(_: widgets.Button) -> None:
    output.clear_output()
    with output:
        rulesets_to_delete = list(ruleset_select.value)
        data_products_to_delete = list(data_product_select.value)
        if not rulesets_to_delete and not data_products_to_delete:
            print("Nothing selected.")
            return
        if rulesets_to_delete:
            delete_rulesets(rulesets_to_delete, dry_run=dry_run_box.value)
        if data_products_to_delete:
            delete_data_products(data_products_to_delete, dry_run=dry_run_box.value)
        if not dry_run_box.value:
            print("\nRemaining in CDF:")
            refresh()


delete_button.on_click(on_delete)
display(widgets.VBox([ruleset_select, data_product_select, dry_run_box, delete_button, output]))
with output:
    refresh()

## Clean up stuck monitor schedules

After switching views (for example from the sample `YourOrg*` views to Cognite Core), a short-lived **Monitor `{view}` validation** function schedule can keep firing every ~2 minutes against an old orchestration job.

If none exist, the next cell only reports that and skips the delete UI. Otherwise select schedules to remove — keep **Dry run** checked to review, then uncheck it and click **Delete selected** again.

In [ ]:
def list_monitor_schedules() -> list:
    """Print and return function schedules whose name starts with 'Monitor '."""
    schedules = [
        schedule
        for schedule in client.functions.schedules.list(limit=None)
        if (schedule.name or "").startswith("Monitor ")
    ]
    for schedule in schedules:
        print(f"id={schedule.id} | {schedule.name} | cron={schedule.cron_expression} | {schedule.description or ''}")
    print(f"{len(schedules)} monitor schedule(s)")
    return schedules


def delete_monitor_schedules(schedule_ids: list[int], dry_run: bool = True) -> None:
    """Delete function schedules by id (typically stuck Monitor `{view}` schedules).

    Args:
        schedule_ids: Function schedule ids from `list_monitor_schedules()`.
        dry_run: Only print what would be deleted.
    """
    if not schedule_ids:
        print("No schedules selected.")
        return
    for schedule_id in schedule_ids:
        schedule = client.functions.schedules.retrieve(id=schedule_id)
        label = f"id={schedule_id}"
        if schedule is not None:
            label = f"id={schedule_id} | {schedule.name} | {schedule.description or ''}"
        if not dry_run:
            client.functions.schedules.delete(id=schedule_id)
        print(f"{_prefix(dry_run)} monitor schedule {label}")


schedules = list_monitor_schedules()
if not schedules:
    print("Nothing to delete.")
else:
    monitor_layout = widgets.Layout(width="95%", height="120px")
    monitor_style = {"description_width": "110px"}
    monitor_select = widgets.SelectMultiple(
        description="Monitors",
        layout=monitor_layout,
        style=monitor_style,
        options=[
            (f"id={schedule.id} | {schedule.name} | {schedule.description or ''}", schedule.id)
            for schedule in schedules
        ],
    )
    monitor_dry_run = widgets.Checkbox(
        value=True, description="Dry run (only show what would be deleted)", style=monitor_style
    )
    monitor_delete_button = widgets.Button(description="Delete selected", button_style="danger", icon="trash")
    monitor_output = widgets.Output()

    def refresh_monitors() -> None:
        """List monitor schedules in CDF and offer them for selection."""
        remaining = list_monitor_schedules()
        monitor_select.options = [
            (f"id={schedule.id} | {schedule.name} | {schedule.description or ''}", schedule.id)
            for schedule in remaining
        ]
        if not remaining:
            print("Nothing to delete.")

    def on_delete_monitors(_: widgets.Button) -> None:
        monitor_output.clear_output()
        with monitor_output:
            selected = list(monitor_select.value)
            delete_monitor_schedules(selected, dry_run=monitor_dry_run.value)
            if not monitor_dry_run.value:
                print("\nRemaining monitor schedules:")
                refresh_monitors()

    monitor_delete_button.on_click(on_delete_monitors)
    display(widgets.VBox([monitor_select, monitor_dry_run, monitor_delete_button, monitor_output]))

## Clean up stale data-quality workflows

`deploy_infrastructure.py` creates new workflows when the data product or views change, but it does **not** remove the previous ones. Old workflows still have triggers with the previous view ids baked into their input (for example `YourOrgAsset` / `enterprise-process-industry`).

This cell lists instance and time-series validation workflows (`dq-shacl-*`, `dq-ts-shacl-*`). Deleting a workflow removes its triggers first, then the workflow. Keep shared workflows (`dq-data-product-sync`, `dq-historic-queue-manager`) and the ones that match your current data product. Keep **Dry run** checked to review, then uncheck it and click **Delete selected** again.

In [ ]:
DQ_WORKFLOW_PREFIXES = ("dq-shacl-", "dq-ts-shacl-")


def _is_dq_validation_workflow(external_id: str) -> bool:
    return external_id.startswith(DQ_WORKFLOW_PREFIXES)


def list_dq_workflows() -> list:
    """Print and return instance/time-series DQ validation workflows with their triggers."""
    workflows = [
        workflow for workflow in client.workflows.list(limit=None) if _is_dq_validation_workflow(workflow.external_id)
    ]
    triggers = list(client.workflows.triggers.list(limit=None))
    triggers_by_workflow: dict[str, list] = {}
    for trigger in triggers:
        workflow_id = trigger.workflow_external_id
        triggers_by_workflow.setdefault(workflow_id, []).append(trigger)

    for workflow in workflows:
        workflow_triggers = triggers_by_workflow.get(workflow.external_id, [])
        print(f"{workflow.external_id} | triggers={len(workflow_triggers)}")
        for trigger in workflow_triggers:
            print(f"    {trigger.external_id}")
    print(f"{len(workflows)} DQ validation workflow(s)")
    return workflows


def delete_dq_workflows(external_ids: list[str], dry_run: bool = True) -> None:
    """Delete DQ validation workflows and every trigger that targets them.

    Args:
        external_ids: Workflow external ids from `list_dq_workflows()`.
        dry_run: Only print what would be deleted.
    """
    if not external_ids:
        print("No workflows selected.")
        return

    selected = set(external_ids)
    triggers = [
        trigger for trigger in client.workflows.triggers.list(limit=None) if trigger.workflow_external_id in selected
    ]
    for trigger in triggers:
        if not dry_run:
            client.workflows.triggers.delete(external_id=trigger.external_id)
        print(f"{_prefix(dry_run)} trigger {trigger.external_id} (workflow={trigger.workflow_external_id})")

    for external_id in external_ids:
        if not dry_run:
            client.workflows.delete(external_id=external_id, ignore_unknown_ids=True)
        print(f"{_prefix(dry_run)} workflow {external_id}")


workflow_layout = widgets.Layout(width="95%", height="180px")
workflow_style = {"description_width": "110px"}
workflow_select = widgets.SelectMultiple(description="Workflows", layout=workflow_layout, style=workflow_style)
workflow_dry_run = widgets.Checkbox(
    value=True, description="Dry run (only show what would be deleted)", style=workflow_style
)
workflow_delete_button = widgets.Button(description="Delete selected", button_style="danger", icon="trash")
workflow_output = widgets.Output()


def refresh_workflows() -> None:
    """List DQ validation workflows in CDF and offer them for selection."""
    workflows = list_dq_workflows()
    triggers = list(client.workflows.triggers.list(limit=None))
    trigger_count = {}
    for trigger in triggers:
        trigger_count[trigger.workflow_external_id] = trigger_count.get(trigger.workflow_external_id, 0) + 1
    workflow_select.options = [
        (
            f"{workflow.external_id} | triggers={trigger_count.get(workflow.external_id, 0)}",
            workflow.external_id,
        )
        for workflow in workflows
    ]


def on_delete_workflows(_: widgets.Button) -> None:
    workflow_output.clear_output()
    with workflow_output:
        selected = list(workflow_select.value)
        delete_dq_workflows(selected, dry_run=workflow_dry_run.value)
        if not workflow_dry_run.value:
            print("\nRemaining DQ validation workflows:")
            refresh_workflows()


workflow_delete_button.on_click(on_delete_workflows)
display(widgets.VBox([workflow_select, workflow_dry_run, workflow_delete_button, workflow_output]))
with workflow_output:
    refresh_workflows()

## Clean up stuck historic queue jobs

After switching views (for example from `YourOrg*` to Cognite Core), `HistoricJobQueue` nodes in `dataQualitySpace` can keep the historic queue manager busy — for example `hq_YourOrgAsset_1_1_10` stuck in `running` / `queued`.

`cdf clean` / redeploy does **not** remove these DMS nodes. Run the setup and helpers cells above first, then this cell. It lists every `HistoricJobQueueView` job. Deleting a selected job also removes its linked `OrchestrationState` (`orch_state_{orchestration_id}`) and any matching **Monitor `{view}` validation** schedule. Keep **Dry run** checked to review, then uncheck it and click **Delete selected** again.

In [ ]:
from cognite.client.data_classes.data_modeling.ids import ViewId
from cognite.client.exceptions import CogniteAPIError

HISTORIC_JOB_QUEUE_VIEW = "HistoricJobQueueView"
HISTORIC_JOB_QUEUE_VIEW_VERSION = "1"

if "client" not in globals() or "_prefix" not in globals():
    raise RuntimeError("Run the setup cell (credentials / client) and the helpers cell first.")


def _resolve_data_quality_space() -> str:
    """Return dataQualitySpace from setup cell, or resolve it from Toolkit config."""
    existing = globals().get("DATA_QUALITY_SPACE")
    if isinstance(existing, str) and existing:
        return existing
    project_root = find_toolkit_project_root()
    toolkit_config = None
    if project_root is not None:
        for name in ("config.dev.yaml", "config.local.yaml", "config.prod.yaml"):
            candidate = project_root / name
            if candidate.is_file():
                toolkit_config = candidate
                break
    return load_variables(toolkit_config).get("dataQualitySpace") or "dataQuality"


DATA_QUALITY_SPACE = _resolve_data_quality_space()
print(f"dataQualitySpace: {DATA_QUALITY_SPACE}")


def _historic_queue_view_id(space: str | None = None) -> ViewId:
    return ViewId(
        space=space or DATA_QUALITY_SPACE,
        external_id=HISTORIC_JOB_QUEUE_VIEW,
        version=HISTORIC_JOB_QUEUE_VIEW_VERSION,
    )


def _historic_job_label(job: dict) -> str:
    return (
        f"{job['external_id']} | status={job.get('status') or '?'} | "
        f"view={job.get('view_external_id') or '?'} | "
        f"dp={job.get('data_product_external_id') or '?'}:{job.get('data_product_version') or '?'} | "
        f"orch={job.get('orchestration_id') or '-'}"
    )


def list_historic_queue_jobs(space: str | None = None) -> list[dict]:
    """Print and return HistoricJobQueue nodes in the data quality space."""
    resolved_space = space or DATA_QUALITY_SPACE
    view_id = _historic_queue_view_id(resolved_space)
    try:
        nodes = client.data_modeling.instances.list(
            instance_type="node",
            sources=[view_id],
            limit=None,
        )
    except CogniteAPIError as exc:
        print(f"Could not list HistoricJobQueue in space {resolved_space}: {exc}")
        return []

    jobs: list[dict] = []
    for node in nodes:
        props = dict(node.properties.get(view_id) or {}) if node.properties else {}
        job = {"external_id": node.external_id, "space": node.space, **props}
        jobs.append(job)
        print(_historic_job_label(job))
    print(f"{len(jobs)} historic queue job(s) in space {resolved_space}")
    return jobs


def _delete_monitor_for_orchestration(view_external_id: str, orchestration_id: str, dry_run: bool) -> None:
    schedule_name = f"Monitor {view_external_id} validation"
    for schedule in client.functions.schedules.list(limit=None):
        if (schedule.name or "") != schedule_name:
            continue
        input_data = client.functions.schedules.get_input_data(id=schedule.id) or {}
        schedule_orch_id = input_data.get("orchestration_id")
        if schedule_orch_id and schedule_orch_id != orchestration_id:
            print(
                f"    Leaving monitor schedule id={schedule.id} "
                f"(tracks {schedule_orch_id}, not {orchestration_id})"
            )
            return
        if not dry_run:
            client.functions.schedules.delete(id=schedule.id)
        print(f"    {_prefix(dry_run)} monitor schedule id={schedule.id} | {schedule_name}")
        return


def delete_historic_queue_jobs(
    external_ids: list[str],
    dry_run: bool = True,
    space: str | None = None,
) -> None:
    """Delete HistoricJobQueue nodes and linked orchestration state / monitor schedules.

    Args:
        external_ids: Queue job external ids from `list_historic_queue_jobs()`.
        dry_run: Only print what would be deleted.
        space: DMS space holding HistoricJobQueue / OrchestrationState.
    """
    if not external_ids:
        print("No historic queue jobs selected.")
        return

    resolved_space = space or DATA_QUALITY_SPACE
    view_id = _historic_queue_view_id(resolved_space)
    for external_id in external_ids:
        result = client.data_modeling.instances.retrieve(
            nodes=[(resolved_space, external_id)], sources=[view_id]
        )
        props: dict[str, object] = {}
        if result.nodes:
            props = dict(result.nodes[0].properties.get(view_id) or {})
        job = {"external_id": external_id, **props}
        print(_historic_job_label(job))

        orchestration_id = props.get("orchestration_id")
        view_external_id = props.get("view_external_id")
        if orchestration_id and view_external_id:
            _delete_monitor_for_orchestration(str(view_external_id), str(orchestration_id), dry_run)
            orch_state_id = f"orch_state_{orchestration_id}"
            if not dry_run:
                client.data_modeling.instances.delete(nodes=[(resolved_space, orch_state_id)])
            print(f"    {_prefix(dry_run)} orchestration state {orch_state_id}")

        if not dry_run:
            client.data_modeling.instances.delete(nodes=[(resolved_space, external_id)])
        print(f"{_prefix(dry_run)} historic queue job {external_id}")


queue_jobs = list_historic_queue_jobs()
if not queue_jobs:
    print("Nothing to delete.")
else:
    queue_layout = widgets.Layout(width="95%", height="180px")
    queue_style = {"description_width": "110px"}
    queue_select = widgets.SelectMultiple(
        description="Queue jobs",
        layout=queue_layout,
        style=queue_style,
        options=[(_historic_job_label(job), job["external_id"]) for job in queue_jobs],
    )
    queue_dry_run = widgets.Checkbox(
        value=True, description="Dry run (only show what would be deleted)", style=queue_style
    )
    queue_delete_button = widgets.Button(description="Delete selected", button_style="danger", icon="trash")
    queue_output = widgets.Output()

    def refresh_queue_jobs() -> None:
        """List historic queue jobs in CDF and offer them for selection."""
        remaining = list_historic_queue_jobs()
        queue_select.options = [(_historic_job_label(job), job["external_id"]) for job in remaining]
        if not remaining:
            print("Nothing to delete.")

    def on_delete_queue_jobs(_: widgets.Button) -> None:
        queue_output.clear_output()
        with queue_output:
            selected = list(queue_select.value)
            delete_historic_queue_jobs(selected, dry_run=queue_dry_run.value)
            if not queue_dry_run.value:
                print("\nRemaining historic queue jobs:")
                refresh_queue_jobs()

    queue_delete_button.on_click(on_delete_queue_jobs)
    display(widgets.VBox([queue_select, queue_dry_run, queue_delete_button, queue_output]))